In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss
)

from xgboost import XGBClassifier

df = pd.read_csv(
    "../data/raw/Case_Study_2_CLV_Predictive_Analytics.csv"
)

print("Dataset:", df.shape)

Dataset: (2000, 9)


In [2]:
TARGET = "churn_event_next_12m"

FEATURES = [
    "current_capacity_tb",
    "utilization_rate_pct",
    "expansion_potential_signal",
    "utilization_above_100_flag"
]

labeled_df = df[df[TARGET].notna()].copy()

labeled_df["utilization_above_100_flag"] = (
    labeled_df["utilization_rate_pct"] > 1.0
).astype(int)

split1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    split1.split(
        labeled_df,
        groups=labeled_df["account_id"]
    )
)

train_df = labeled_df.iloc[train_idx].copy()
temp_df = labeled_df.iloc[temp_idx].copy()

split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    split2.split(
        temp_df,
        groups=temp_df["account_id"]
    )
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

X_train = train_df[FEATURES]
y_train = train_df[TARGET].astype(int)

X_val = val_df[FEATURES]
y_val = val_df[TARGET].astype(int)

X_test = test_df[FEATURES]
y_test = test_df[TARGET].astype(int)

assert set(train_df.account_id).isdisjoint(val_df.account_id)
assert set(train_df.account_id).isdisjoint(test_df.account_id)
assert set(val_df.account_id).isdisjoint(test_df.account_id)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("No account leakage detected.")

Train: (1328, 4)
Validation: (280, 4)
Test: (276, 4)
No account leakage detected.


In [3]:
baseline_val_prob = np.full(
    len(y_val),
    0.05
)

baseline_results = {
    "Model": "Static 5% Baseline",
    "ROC-AUC": 0.5,
    "PR-AUC": average_precision_score(
        y_val, baseline_val_prob
    ),
    "Brier Score": brier_score_loss(
        y_val, baseline_val_prob
    ),
    "Log Loss": log_loss(
        y_val, baseline_val_prob,
        labels=[0, 1]
    ),
    "Mean Predicted Churn": baseline_val_prob.mean(),
    "Observed Churn": y_val.mean()
}

display(pd.DataFrame([baseline_results]).round(4))

,Model,ROC-AUC,PR-AUC,Brier Score,Log Loss,Mean Predicted Churn,Observed Churn
0,Static 5% Baseline,0.5,0.1679,0.1536,0.5455,0.05,0.1679


In [4]:
numeric_features = [
    "current_capacity_tb",
    "utilization_rate_pct",
    "utilization_above_100_flag"
]

categorical_features = [
    "expansion_potential_signal"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [5]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_val_prob = rf_pipeline.predict_proba(X_val)[:, 1]

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [6]:
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_pipeline.fit(X_train, y_train)

xgb_val_prob = xgb_pipeline.predict_proba(X_val)[:, 1]

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [7]:
def evaluate_model(name, y_true, probabilities):
    return {
        "Model": name,
        "ROC-AUC": roc_auc_score(
            y_true, probabilities
        ),
        "PR-AUC": average_precision_score(
            y_true, probabilities
        ),
        "Brier Score": brier_score_loss(
            y_true, probabilities
        ),
        "Log Loss": log_loss(
            y_true, probabilities,
            labels=[0, 1]
        ),
        "Mean Predicted Churn": probabilities.mean(),
        "Observed Churn": y_true.mean()
    }

results = pd.DataFrame([
    baseline_results,
    evaluate_model(
        "Random Forest", y_val, rf_val_prob
    ),
    evaluate_model(
        "XGBoost", y_val, xgb_val_prob
    )
])

display(
    results.sort_values("Brier Score").round(4)
)

,Model,ROC-AUC,PR-AUC,Brier Score,Log Loss,Mean Predicted Churn,Observed Churn
1,Random Forest,0.5452,0.1950,0.1404,0.4552,0.1609,0.1679
2,XGBoost,0.5359,0.1888,0.1471,0.4813,0.1529,0.1679
0,Static 5% Baseline,0.5000,0.1679,0.1536,0.5455,0.0500,0.1679


In [8]:
# Baseline based only on observed training churn
train_churn_rate = y_train.mean()

prevalence_val_prob = np.full(
    len(y_val),
    train_churn_rate
)

prevalence_results = evaluate_model(
    "Training Prevalence Baseline",
    y_val,
    prevalence_val_prob
)

print("Training churn rate:", round(train_churn_rate * 100, 2), "%")

display(
    pd.DataFrame([
        baseline_results,
        prevalence_results,
        evaluate_model("Random Forest", y_val, rf_val_prob),
        evaluate_model("XGBoost", y_val, xgb_val_prob)
    ]).sort_values("Brier Score").round(4)
)

Training churn rate: 16.11 %


,Model,ROC-AUC,PR-AUC,Brier Score,Log Loss,Mean Predicted Churn,Observed Churn
1,Training Prevalence Baseline,0.5000,0.1679,0.1397,0.4526,0.1611,0.1679
2,Random Forest,0.5452,0.1950,0.1404,0.4552,0.1609,0.1679
3,XGBoost,0.5359,0.1888,0.1471,0.4813,0.1529,0.1679
0,Static 5% Baseline,0.5000,0.1679,0.1536,0.5455,0.0500,0.1679


## Initial Model Comparison

Four approaches were evaluated:

1. Static 5% churn baseline
2. Training-prevalence baseline
3. Random Forest classifier
4. XGBoost classifier

### Initial Findings

Random Forest and XGBoost improve upon the existing
5% churn assumption on the validation set.

However, the training-prevalence baseline achieves
the lowest Brier score and log loss.

This suggests that correcting the overall churn
assumption may be more valuable than the current
models' ability to distinguish individual arrays.

Further grouped cross-validation and targeted model
experiments are needed before final model selection.